# Training Visualization and Monitoring

This notebook helps you visualize training progress and understand what the model is learning.


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import sys
import os
from torch.utils.tensorboard import SummaryWriter
from pathlib import Path

sys.path.append('..')

from src.models.unet import UNet
from src.models.diffusion import DiffusionModel
from src.utils.visualization import save_samples, plot_training_curves


## 1. Load Trained Model (if available)

First, let's check if you have any trained checkpoints:


In [ ]:
# Check for checkpoints
checkpoint_dir = '../outputs/checkpoints'
checkpoints = []

if os.path.exists(checkpoint_dir):
    checkpoints = sorted([f for f in os.listdir(checkpoint_dir) if f.endswith('.pt')])
    print(f"Found {len(checkpoints)} checkpoint(s):")
    for i, ckpt in enumerate(checkpoints):
        print(f"  {i+1}. {ckpt}")
else:
    print(f"Checkpoint directory {checkpoint_dir} not found.")
    print("Train a model first using: python scripts/train.py")

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nUsing device: {device}")


## 2. Train Models

Choose one of the model configurations below to train. Each cell is self-contained and will:
- Load the dataset
- Create a model with the specified configuration
- Train the model
- Save checkpoints automatically

**Note**: Training can take a long time. Start with the small model for testing.


### Option 1: Small Model (Fast Training - Good for Testing)

**Configuration:**
- Model channels: 64
- Channel multipliers: (1, 2, 4)
- Parameters: ~7M
- Training time: ~30-60 min per epoch (on GPU)

**Use this for:** Quick testing, learning, or limited compute resources.


In [ ]:
# ============================================
# TRAIN SMALL MODEL
# ============================================
# Uncomment and run this cell to train the small model

from torch.utils.data import DataLoader
from src.data.dataset import ImageDataset
from src.training.trainer import Trainer

# Training configuration
TRAIN_CONFIG = {
    'data_dir': '../data/processed',
    'image_size': 64,
    'batch_size': 32,
    'num_epochs': 50,  # Adjust as needed
    'learning_rate': 1e-4,
    'output_dir': '../outputs',
    'num_workers': 0,  # Set to 0 for Windows, 2-4 for Linux/Mac
}

# Model configuration (SMALL)
MODEL_CONFIG_SMALL = {
    'in_channels': 3,
    'out_channels': 3,
    'time_emb_dim': 64,
    'model_channels': 64,
    'channel_mult': (1, 2, 4),
    'attention_resolutions': (16,),
    'dropout': 0.1,
}

# Diffusion configuration
DIFFUSION_CONFIG = {
    'timesteps': 1000,
    'schedule_type': 'linear',  # or 'cosine'
}

# ============================================
# Setup
# ============================================
print("=" * 60)
print("TRAINING SMALL MODEL")
print("=" * 60)

# Check if data exists
if not os.path.exists(TRAIN_CONFIG['data_dir']):
    print(f"ERROR: Data directory {TRAIN_CONFIG['data_dir']} not found!")
    print("Please run notebook 02-data-exploration.ipynb first to download and prepare data.")
    raise FileNotFoundError(f"Data directory not found: {TRAIN_CONFIG['data_dir']}")

# Load dataset
print(f"\nLoading dataset from {TRAIN_CONFIG['data_dir']}...")
dataset = ImageDataset(
    root_dir=TRAIN_CONFIG['data_dir'],
    image_size=TRAIN_CONFIG['image_size']
)

dataloader = DataLoader(
    dataset,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=True,
    num_workers=TRAIN_CONFIG['num_workers'],
    pin_memory=False
)

print(f"Dataset size: {len(dataset):,} images")
print(f"Number of batches: {len(dataloader):,}")

# Create model
print("\nCreating model...")
unet = UNet(**MODEL_CONFIG_SMALL)
total_params = sum(p.numel() for p in unet.parameters())
print(f"Model parameters: {total_params:,}")

# Create diffusion model
print("Creating diffusion model...")
diffusion_model = DiffusionModel(
    model=unet,
    timesteps=DIFFUSION_CONFIG['timesteps'],
    schedule_type=DIFFUSION_CONFIG['schedule_type'],
    device=device
)

# Create trainer
print("\nSetting up trainer...")
trainer = Trainer(
    model=diffusion_model,
    dataloader=dataloader,
    num_epochs=TRAIN_CONFIG['num_epochs'],
    lr=TRAIN_CONFIG['learning_rate'],
    device=device,
    output_dir=TRAIN_CONFIG['output_dir']
)

# Start training
print("\n" + "=" * 60)
print("Starting training...")
print("=" * 60)
trainer.train()

print("\n" + "=" * 60)
print("Training completed!")
print("=" * 60)
print(f"\nCheckpoints saved to: {TRAIN_CONFIG['output_dir']}/checkpoints")
print(f"TensorBoard logs: {TRAIN_CONFIG['output_dir']}/logs")
print("\nTo view training progress:")
print("  tensorboard --logdir=outputs/logs")


### Option 2: Medium Model (Default - Balanced)

**Configuration:**
- Model channels: 128
- Channel multipliers: (1, 2, 4, 8)
- Parameters: ~117M
- Training time: ~1-2 hours per epoch (on GPU)

**Use this for:** Standard training with good quality/speed balance.


In [ ]:
# ============================================
# TRAIN MEDIUM MODEL (DEFAULT)
# ============================================
# Uncomment and run this cell to train the medium model

from torch.utils.data import DataLoader
from src.data.dataset import ImageDataset
from src.training.trainer import Trainer

# Training configuration
TRAIN_CONFIG = {
    'data_dir': '../data/processed',
    'image_size': 64,
    'batch_size': 16,  # Smaller batch for larger model
    'num_epochs': 100,  # Adjust as needed
    'learning_rate': 1e-4,
    'output_dir': '../outputs',
    'num_workers': 0,  # Set to 0 for Windows, 2-4 for Linux/Mac
}

# Model configuration (MEDIUM - DEFAULT)
MODEL_CONFIG_MEDIUM = {
    'in_channels': 3,
    'out_channels': 3,
    'time_emb_dim': 128,
    'model_channels': 128,
    'channel_mult': (1, 2, 4, 8),
    'attention_resolutions': (16,),
    'dropout': 0.1,
}

# Diffusion configuration
DIFFUSION_CONFIG = {
    'timesteps': 1000,
    'schedule_type': 'linear',  # or 'cosine'
}

# ============================================
# Setup
# ============================================
print("=" * 60)
print("TRAINING MEDIUM MODEL (DEFAULT)")
print("=" * 60)

# Check if data exists
if not os.path.exists(TRAIN_CONFIG['data_dir']):
    print(f"ERROR: Data directory {TRAIN_CONFIG['data_dir']} not found!")
    print("Please run notebook 02-data-exploration.ipynb first to download and prepare data.")
    raise FileNotFoundError(f"Data directory not found: {TRAIN_CONFIG['data_dir']}")

# Load dataset
print(f"\nLoading dataset from {TRAIN_CONFIG['data_dir']}...")
dataset = ImageDataset(
    root_dir=TRAIN_CONFIG['data_dir'],
    image_size=TRAIN_CONFIG['image_size']
)

dataloader = DataLoader(
    dataset,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=True,
    num_workers=TRAIN_CONFIG['num_workers'],
    pin_memory=False
)

print(f"Dataset size: {len(dataset):,} images")
print(f"Number of batches: {len(dataloader):,}")

# Create model
print("\nCreating model...")
unet = UNet(**MODEL_CONFIG_MEDIUM)
total_params = sum(p.numel() for p in unet.parameters())
print(f"Model parameters: {total_params:,}")

# Create diffusion model
print("Creating diffusion model...")
diffusion_model = DiffusionModel(
    model=unet,
    timesteps=DIFFUSION_CONFIG['timesteps'],
    schedule_type=DIFFUSION_CONFIG['schedule_type'],
    device=device
)

# Create trainer
print("\nSetting up trainer...")
trainer = Trainer(
    model=diffusion_model,
    dataloader=dataloader,
    num_epochs=TRAIN_CONFIG['num_epochs'],
    lr=TRAIN_CONFIG['learning_rate'],
    device=device,
    output_dir=TRAIN_CONFIG['output_dir']
)

# Start training
print("\n" + "=" * 60)
print("Starting training...")
print("=" * 60)
trainer.train()

print("\n" + "=" * 60)
print("Training completed!")
print("=" * 60)
print(f"\nCheckpoints saved to: {TRAIN_CONFIG['output_dir']}/checkpoints")
print(f"TensorBoard logs: {TRAIN_CONFIG['output_dir']}/logs")
print("\nTo view training progress:")
print("  tensorboard --logdir=outputs/logs")


### Option 3: Large Model (High Quality - Slower)

**Configuration:**
- Model channels: 256
- Channel multipliers: (1, 1, 2, 2, 4, 4)
- Parameters: ~400M+
- Training time: ~3-5 hours per epoch (on GPU)

**Use this for:** Best quality results, requires more GPU memory and time.

**Note**: You may need to reduce batch_size further if you run out of memory.


In [ ]:
# ============================================
# TRAIN LARGE MODEL
# ============================================
# Uncomment and run this cell to train the large model
# WARNING: Requires significant GPU memory and time

from torch.utils.data import DataLoader
from src.data.dataset import ImageDataset
from src.training.trainer import Trainer

# Training configuration
TRAIN_CONFIG = {
    'data_dir': '../data/processed',
    'image_size': 64,
    'batch_size': 8,  # Smaller batch for large model
    'num_epochs': 100,  # Adjust as needed
    'learning_rate': 5e-5,  # Slightly lower LR for larger model
    'output_dir': '../outputs',
    'num_workers': 0,  # Set to 0 for Windows, 2-4 for Linux/Mac
}

# Model configuration (LARGE)
MODEL_CONFIG_LARGE = {
    'in_channels': 3,
    'out_channels': 3,
    'time_emb_dim': 256,
    'model_channels': 256,
    'channel_mult': (1, 1, 2, 2, 4, 4),
    'attention_resolutions': (32, 16, 8),  # More attention layers
    'dropout': 0.1,
}

# Diffusion configuration
DIFFUSION_CONFIG = {
    'timesteps': 1000,
    'schedule_type': 'cosine',  # Cosine schedule often works better for larger models
}

# ============================================
# Setup
# ============================================
print("=" * 60)
print("TRAINING LARGE MODEL")
print("=" * 60)
print("WARNING: This model requires significant GPU memory!")
print("=" * 60)

# Check if data exists
if not os.path.exists(TRAIN_CONFIG['data_dir']):
    print(f"ERROR: Data directory {TRAIN_CONFIG['data_dir']} not found!")
    print("Please run notebook 02-data-exploration.ipynb first to download and prepare data.")
    raise FileNotFoundError(f"Data directory not found: {TRAIN_CONFIG['data_dir']}")

# Load dataset
print(f"\nLoading dataset from {TRAIN_CONFIG['data_dir']}...")
dataset = ImageDataset(
    root_dir=TRAIN_CONFIG['data_dir'],
    image_size=TRAIN_CONFIG['image_size']
)

dataloader = DataLoader(
    dataset,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=True,
    num_workers=TRAIN_CONFIG['num_workers'],
    pin_memory=False
)

print(f"Dataset size: {len(dataset):,} images")
print(f"Number of batches: {len(dataloader):,}")

# Create model
print("\nCreating model...")
unet = UNet(**MODEL_CONFIG_LARGE)
total_params = sum(p.numel() for p in unet.parameters())
print(f"Model parameters: {total_params:,}")

# Check GPU memory
if device == 'cuda':
    import torch
    free_memory = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
    print(f"Available GPU memory: {free_memory / 1e9:.2f} GB")
    if free_memory < 8e9:  # Less than 8GB
        print("⚠ WARNING: Low GPU memory. Consider using a smaller model or batch size.")

# Create diffusion model
print("Creating diffusion model...")
diffusion_model = DiffusionModel(
    model=unet,
    timesteps=DIFFUSION_CONFIG['timesteps'],
    schedule_type=DIFFUSION_CONFIG['schedule_type'],
    device=device
)

# Create trainer
print("\nSetting up trainer...")
trainer = Trainer(
    model=diffusion_model,
    dataloader=dataloader,
    num_epochs=TRAIN_CONFIG['num_epochs'],
    lr=TRAIN_CONFIG['learning_rate'],
    device=device,
    output_dir=TRAIN_CONFIG['output_dir']
)

# Start training
print("\n" + "=" * 60)
print("Starting training...")
print("=" * 60)
trainer.train()

print("\n" + "=" * 60)
print("Training completed!")
print("=" * 60)
print(f"\nCheckpoints saved to: {TRAIN_CONFIG['output_dir']}/checkpoints")
print(f"TensorBoard logs: {TRAIN_CONFIG['output_dir']}/logs")
print("\nTo view training progress:")
print("  tensorboard --logdir=outputs/logs")


### Option 4: Custom Model Configuration

Create your own model configuration by modifying the parameters below.


In [ ]:
# ============================================
# TRAIN CUSTOM MODEL
# ============================================
# Modify the configurations below to create your own model

from torch.utils.data import DataLoader
from src.data.dataset import ImageDataset
from src.training.trainer import Trainer

# ============================================
# CUSTOMIZE THESE PARAMETERS
# ============================================

# Training configuration
TRAIN_CONFIG = {
    'data_dir': '../data/processed',
    'image_size': 64,  # Image size (32, 64, 128, etc.)
    'batch_size': 16,  # Adjust based on GPU memory
    'num_epochs': 50,  # Number of training epochs
    'learning_rate': 1e-4,  # Learning rate (try 1e-4, 5e-5, or 2e-4)
    'output_dir': '../outputs',
    'num_workers': 0,  # Data loading workers (0 for Windows)
}

# Model configuration - CUSTOMIZE THIS
MODEL_CONFIG_CUSTOM = {
    'in_channels': 3,
    'out_channels': 3,
    'time_emb_dim': 128,  # Time embedding dimension
    'model_channels': 128,  # Base number of channels
    'channel_mult': (1, 2, 4, 8),  # Channel multipliers at each resolution
    'attention_resolutions': (16,),  # Resolutions to apply attention
    'dropout': 0.1,  # Dropout rate
}

# Diffusion configuration
DIFFUSION_CONFIG = {
    'timesteps': 1000,  # Number of diffusion timesteps
    'schedule_type': 'linear',  # 'linear' or 'cosine'
}

# ============================================
# Setup and Training
# ============================================
print("=" * 60)
print("TRAINING CUSTOM MODEL")
print("=" * 60)

# Check if data exists
if not os.path.exists(TRAIN_CONFIG['data_dir']):
    print(f"ERROR: Data directory {TRAIN_CONFIG['data_dir']} not found!")
    print("Please run notebook 02-data-exploration.ipynb first to download and prepare data.")
    raise FileNotFoundError(f"Data directory not found: {TRAIN_CONFIG['data_dir']}")

# Load dataset
print(f"\nLoading dataset from {TRAIN_CONFIG['data_dir']}...")
dataset = ImageDataset(
    root_dir=TRAIN_CONFIG['data_dir'],
    image_size=TRAIN_CONFIG['image_size']
)

dataloader = DataLoader(
    dataset,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=True,
    num_workers=TRAIN_CONFIG['num_workers'],
    pin_memory=False
)

print(f"Dataset size: {len(dataset):,} images")
print(f"Number of batches: {len(dataloader):,}")

# Create model
print("\nCreating model...")
print(f"Configuration: {MODEL_CONFIG_CUSTOM}")
unet = UNet(**MODEL_CONFIG_CUSTOM)
total_params = sum(p.numel() for p in unet.parameters())
print(f"Model parameters: {total_params:,}")

# Create diffusion model
print("Creating diffusion model...")
diffusion_model = DiffusionModel(
    model=unet,
    timesteps=DIFFUSION_CONFIG['timesteps'],
    schedule_type=DIFFUSION_CONFIG['schedule_type'],
    device=device
)

# Create trainer
print("\nSetting up trainer...")
trainer = Trainer(
    model=diffusion_model,
    dataloader=dataloader,
    num_epochs=TRAIN_CONFIG['num_epochs'],
    lr=TRAIN_CONFIG['learning_rate'],
    device=device,
    output_dir=TRAIN_CONFIG['output_dir']
)

# Start training
print("\n" + "=" * 60)
print("Starting training...")
print("=" * 60)
trainer.train()

print("\n" + "=" * 60)
print("Training completed!")
print("=" * 60)
print(f"\nCheckpoints saved to: {TRAIN_CONFIG['output_dir']}/checkpoints")
print(f"TensorBoard logs: {TRAIN_CONFIG['output_dir']}/logs")
print("\nTo view training progress:")
print("  tensorboard --logdir=outputs/logs")


## 2. Load and Inspect a Checkpoint


In [ ]:
# Load a checkpoint
if checkpoints:
    checkpoint_path = os.path.join(checkpoint_dir, checkpoints[-1])  # Load latest
    print(f"Loading checkpoint: {checkpoint_path}")
    
    # Create model (adjust parameters to match your training config)
    model = UNet(
        in_channels=3,
        out_channels=3,
        time_emb_dim=128,
        model_channels=128,
        channel_mult=(1, 2, 4, 8),
        attention_resolutions=(16,),
        dropout=0.1
    )
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    
    print(f"✓ Loaded checkpoint from epoch {checkpoint.get('epoch', 'unknown')}")
    print(f"✓ Training loss: {checkpoint.get('loss', 'unknown'):.4f}")
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"✓ Total parameters: {total_params:,}")
    print(f"✓ Trainable parameters: {trainable_params:,}")
else:
    print("No checkpoints found. Please train a model first.")


## 3. Generate and Visualize Samples


In [ ]:
if 'model' in locals():
    # Create diffusion model
    diffusion_model = DiffusionModel(
        model=model,
        timesteps=1000,
        schedule_type='linear',
        device=device
    )
    
    # Generate samples
    print("Generating samples...")
    print("This may take a few minutes (1000 denoising steps)...")
    
    samples = diffusion_model.sample(
        image_size=64,
        batch_size=16,
        channels=3
    )
    
    # Visualize
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for i, ax in enumerate(axes.flat):
        if i < samples.shape[0]:
            img = samples[i].permute(1, 2, 0).cpu().numpy()
            ax.imshow(img)
            ax.axis('off')
    
    plt.tight_layout()
    plt.suptitle('Generated Samples from Trained Model', y=1.02)
    plt.show()
    
    # Save samples
    os.makedirs('../outputs/samples', exist_ok=True)
    save_samples(samples, '../outputs/samples/generated_samples.png', nrow=4)
    print("\n✓ Samples saved to outputs/samples/generated_samples.png")
else:
    print("No model loaded. Please load a checkpoint first.")


## 4. Visualize Training Progress (from TensorBoard logs)

If you have TensorBoard logs, you can visualize training curves:


In [ ]:
# Check for TensorBoard logs
log_dir = '../outputs/logs'

if os.path.exists(log_dir):
    print("TensorBoard logs found!")
    print("\nTo visualize training progress:")
    print("1. Start TensorBoard: tensorboard --logdir=outputs/logs")
    print("2. Open http://localhost:6006 in your browser")
    print("\nOr use TensorBoard programmatically:")
    
    try:
        from torch.utils.tensorboard import SummaryWriter
        from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
        
        # Try to read events
        event_files = []
        for root, dirs, files in os.walk(log_dir):
            for file in files:
                if 'events.out.tfevents' in file:
                    event_files.append(os.path.join(root, file))
        
        if event_files:
            print(f"\nFound {len(event_files)} event file(s)")
            print("Use TensorBoard to view detailed training metrics")
        else:
            print("\nNo event files found yet. Start training to generate logs.")
    except ImportError:
        print("\nTensorBoard not installed. Install with: pip install tensorboard")
else:
    print(f"Log directory {log_dir} not found.")
    print("Training logs will be created when you run training.")


## 5. Compare Different Checkpoints

Compare samples from different training stages to see improvement:


In [ ]:
# Compare samples from different training stages
if len(checkpoints) >= 2:
    # Select checkpoints to compare (early, middle, late)
    indices = [0, len(checkpoints)//2, -1] if len(checkpoints) >= 3 else [0, -1]
    selected_checkpoints = [checkpoints[i] for i in indices]
    
    print(f"Comparing {len(selected_checkpoints)} checkpoints:")
    for ckpt in selected_checkpoints:
        print(f"  - {ckpt}")
    
    fig, axes = plt.subplots(len(selected_checkpoints), 4, 
                            figsize=(12, 3 * len(selected_checkpoints)))
    
    if len(selected_checkpoints) == 1:
        axes = axes.reshape(1, -1)
    
    for epoch_idx, checkpoint_name in enumerate(selected_checkpoints):
        checkpoint_path = os.path.join(checkpoint_dir, checkpoint_name)
        
        # Load model
        model_comp = UNet(
            in_channels=3,
            out_channels=3,
            time_emb_dim=128,
            model_channels=128,
            channel_mult=(1, 2, 4, 8)
        )
        
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model_comp.load_state_dict(checkpoint['model_state_dict'])
        model_comp = model_comp.to(device)
        
        # Generate samples
        diffusion_model_comp = DiffusionModel(
            model_comp, 
            timesteps=1000, 
            schedule_type='linear',
            device=device
        )
        
        # Generate fewer samples for comparison (faster)
        samples_comp = diffusion_model_comp.sample(
            image_size=64, 
            batch_size=4, 
            channels=3
        )
        
        # Visualize
        for i in range(4):
            img = samples_comp[i].permute(1, 2, 0).cpu().numpy()
            axes[epoch_idx, i].imshow(img)
            axes[epoch_idx, i].axis('off')
        
        epoch_num = checkpoint.get('epoch', '?')
        loss = checkpoint.get('loss', '?')
        axes[epoch_idx, 0].set_ylabel(
            f'Epoch {epoch_num}\nLoss: {loss:.4f}', 
            rotation=0, 
            ha='right',
            fontsize=10
        )
    
    plt.tight_layout()
    plt.suptitle('Training Progress: Samples at Different Epochs', y=1.02)
    plt.show()
    
    print("\n✓ Comparison complete!")
    print("You should see improvement in sample quality as training progresses.")
else:
    print("Need at least 2 checkpoints to compare. Train for more epochs.")


## 6. Analyze Model Predictions

Let's see what the model predicts at different noise levels:


In [ ]:
if 'model' in locals():
    # Create a simple test image
    test_image = torch.randn(1, 3, 64, 64).to(device)
    
    # Test at different timesteps
    timesteps_to_test = [0, 200, 400, 600, 800, 999]
    
    model.eval()
    with torch.no_grad():
        fig, axes = plt.subplots(2, len(timesteps_to_test), figsize=(15, 6))
        
        for i, t_val in enumerate(timesteps_to_test):
            t = torch.tensor([t_val], device=device)
            
            # Get prediction
            noise_pred = model(test_image, t)
            
            # Visualize input (noisy image)
            img_in = test_image[0].permute(1, 2, 0).cpu()
            img_in = (img_in + 1) / 2
            axes[0, i].imshow(torch.clamp(img_in, 0, 1).numpy())
            axes[0, i].set_title(f'Input (t={t_val})')
            axes[0, i].axis('off')
            
            # Visualize predicted noise
            noise_vis = noise_pred[0].permute(1, 2, 0).cpu()
            noise_vis = (noise_vis + 1) / 2
            axes[1, i].imshow(torch.clamp(noise_vis, 0, 1).numpy())
            axes[1, i].set_title(f'Predicted Noise')
            axes[1, i].axis('off')
        
        plt.tight_layout()
        plt.suptitle('Model Predictions at Different Timesteps', y=1.02)
        plt.show()
        
        print("Top row: Input noisy images at different timesteps")
        print("Bottom row: Model's predicted noise")
        print("\nThe model should predict noise that matches the input's noise level.")
else:
    print("No model loaded. Please load a checkpoint first.")


## 7. Check Model Architecture

Let's inspect the model structure:


In [ ]:
if 'model' in locals():
    # Print model summary
    print("Model Architecture:")
    print("=" * 50)
    print(model)
    print("=" * 50)
    
    # Count parameters by layer type
    conv_params = 0
    linear_params = 0
    norm_params = 0
    other_params = 0
    
    for name, param in model.named_parameters():
        if 'conv' in name.lower():
            conv_params += param.numel()
        elif 'linear' in name.lower() or 'time_mlp' in name.lower():
            linear_params += param.numel()
        elif 'norm' in name.lower():
            norm_params += param.numel()
        else:
            other_params += param.numel()
    
    print("\nParameter Breakdown:")
    print(f"  Convolutional layers: {conv_params:,} ({conv_params/sum(p.numel() for p in model.parameters())*100:.1f}%)")
    print(f"  Linear/MLP layers: {linear_params:,} ({linear_params/sum(p.numel() for p in model.parameters())*100:.1f}%)")
    print(f"  Normalization layers: {norm_params:,} ({norm_params/sum(p.numel() for p in model.parameters())*100:.1f}%)")
    print(f"  Other: {other_params:,} ({other_params/sum(p.numel() for p in model.parameters())*100:.1f}%)")
    
    # Test forward pass timing
    test_input = torch.randn(1, 3, 64, 64).to(device)
    test_t = torch.randint(0, 1000, (1,)).to(device)
    
    model.eval()
    with torch.no_grad():
        import time
        start = time.time()
        for _ in range(10):
            _ = model(test_input, test_t)
        elapsed = (time.time() - start) / 10
    
    print(f"\nForward pass time: {elapsed*1000:.2f} ms")
else:
    print("No model loaded. Please load a checkpoint first.")


## 8. Generate Multiple Batches

Generate multiple batches to assess diversity:


In [ ]:
if 'model' in locals() and 'diffusion_model' in locals():
    print("Generating multiple batches to check diversity...")
    
    num_batches = 3
    all_samples = []
    
    for batch_idx in range(num_batches):
        print(f"Generating batch {batch_idx + 1}/{num_batches}...")
        samples = diffusion_model.sample(
            image_size=64,
            batch_size=8,
            channels=3
        )
        all_samples.append(samples)
    
    # Combine and visualize
    combined = torch.cat(all_samples, dim=0)
    
    fig, axes = plt.subplots(num_batches, 8, figsize=(16, 3 * num_batches))
    if num_batches == 1:
        axes = axes.reshape(1, -1)
    
    idx = 0
    for batch_idx in range(num_batches):
        for img_idx in range(8):
            img = combined[idx].permute(1, 2, 0).cpu().numpy()
            axes[batch_idx, img_idx].imshow(img)
            axes[batch_idx, img_idx].axis('off')
            idx += 1
    
    plt.tight_layout()
    plt.suptitle(f'Generated Samples from {num_batches} Batches (Diversity Check)', y=1.02)
    plt.show()
    
    print(f"\n✓ Generated {num_batches * 8} samples")
    print("Check for diversity - samples should vary, not be identical.")
else:
    print("No model loaded. Please load a checkpoint first.")


## Tips for Monitoring Training

1. **Loss Curve**: Should decrease steadily, then stabilize
   - If loss increases, learning rate might be too high
   - If loss plateaus early, learning rate might be too low

2. **Sample Quality**: Visual inspection is important
   - Check samples regularly during training
   - Early epochs: Very noisy, unrecognizable
   - Mid training: Some structure appears
   - Late training: Clear, detailed images

3. **FID Score**: Use FID to quantitatively measure quality (optional)
   - Lower FID = better quality
   - Install: `pip install pytorch-fid`

4. **Overfitting**: If loss decreases but samples don't improve
   - Model might be memorizing training data
   - Try data augmentation or regularization

5. **Learning Rate**: Adjust if loss plateaus or becomes unstable
   - Common values: 1e-4 to 5e-5
   - Use learning rate scheduling (cosine annealing)

6. **Checkpoint Management**: Save regularly
   - Keep best model (lowest loss or best FID)
   - Keep checkpoints at different stages for comparison

## Next Steps

- Continue training if samples need improvement
- Experiment with different architectures
- Try advanced techniques from `guides/07-advanced-techniques.md`
- Evaluate with metrics like FID score
